# Grocery Delivery Analytics Pipeline - 02 Silver Processing

This notebook reads the six Bronze Delta tables, profiles their quality problems, applies documented cleaning rules, validates primary/foreign keys, and writes six Silver Delta tables plus a quality summary.

## 1. Setup and reusable quality helpers

In [ ]:
from pyspark.sql import Window
from pyspark.sql import functions as F

CATALOG = "workspace"
SCHEMA = "analytics"
spark.sql(f"USE CATALOG {CATALOG}")
spark.sql(f"USE SCHEMA {SCHEMA}")
spark.conf.set("spark.sql.ansi.enabled", "false")

def read_layer(name, layer):
    return spark.table(f"{CATALOG}.{SCHEMA}.{name}_{layer}")

def write_silver(df, name):
    full_name = f"{CATALOG}.{SCHEMA}.{name}_silver"
    (df.write.format("delta").mode("overwrite")
       .option("overwriteSchema", "true").saveAsTable(full_name))
    print(f"{full_name}: {df.count():,} rows")

def numeric_from_raw(column_name):
    return F.regexp_replace(F.col(column_name).cast("string"), r"[^0-9.\-]", "").cast("double")

def duplicate_count(df, key):
    return df.groupBy(key).count().filter(F.col("count") > 1).count()

bronze_names = ["customers", "stores", "products", "orders", "order_items", "deliveries"]
bronze = {name: read_layer(name, "bronze") for name in bronze_names}
for name, df in bronze.items():
    print(f"{name}: {df.count():,} rows, {len(df.columns)} columns")

## 2. Customers - owner: Hazim Ali

- Deduplicate by `CustomerID`.
- Normalize city, loyalty, and email.
- Parse both date formats.
- Convert age to integer, reject ages outside 18-90, then impute with the median.

In [ ]:
customers_raw = bronze["customers"]
customer_window = Window.partitionBy("CustomerID").orderBy(F.col("RegistrationDateRaw").asc_nulls_last())

customers_stage = (
    customers_raw
    .withColumn("_rn", F.row_number().over(customer_window)).filter(F.col("_rn") == 1).drop("_rn")
    .withColumn("Email", F.lower(F.trim(F.coalesce(F.col("Email"), F.lit("unknown")))))
    .withColumn("City", F.initcap(F.trim(F.col("City"))))
    .withColumn("LoyaltyStatus", F.initcap(F.trim(F.col("LoyaltyStatus"))))
    .withColumn("RegistrationDate", F.coalesce(
        F.to_date("RegistrationDateRaw", "yyyy-MM-dd"),
        F.to_date("RegistrationDateRaw", "MM/dd/yyyy"),
    ))
    .withColumn("Age", F.col("AgeRaw").cast("int"))
    .withColumn("Age", F.when(F.col("Age").between(18, 90), F.col("Age")))
)
median_age = int(customers_stage.approxQuantile("Age", [0.5], 0.01)[0])
customers_silver = (
    customers_stage
    .withColumn("Age", F.coalesce(F.col("Age"), F.lit(median_age)))
    .select("CustomerID", "CustomerFullName", "Email", "City", "LoyaltyStatus", "RegistrationDate", "Age")
)
write_silver(customers_silver, "customers")
display(customers_silver.groupBy("LoyaltyStatus").count().orderBy("LoyaltyStatus"))

## 3. Stores - owner: Mannan

- Normalize city and store type.
- Cast rating to numeric.
- Replace missing/out-of-range ratings with the valid median.

In [ ]:
stores_stage = (
    bronze["stores"].dropDuplicates(["StoreID"])
    .withColumn("City", F.initcap(F.trim(F.col("City"))))
    .withColumn("StoreType", F.initcap(F.trim(F.col("StoreType"))))
    .withColumn("Rating", numeric_from_raw("RatingRaw"))
    .withColumn("Rating", F.when(F.col("Rating").between(1, 5), F.col("Rating")))
)
median_rating = float(stores_stage.approxQuantile("Rating", [0.5], 0.01)[0])
stores_silver = (
    stores_stage.withColumn("Rating", F.coalesce(F.col("Rating"), F.lit(median_rating)))
    .select("StoreID", "StoreName", "City", "StoreType", "Rating")
)
write_silver(stores_silver, "stores")
display(stores_silver.orderBy("StoreID"))

## 4. Products - owner: Maheshwar

- Normalize categories and fill missing category with `Uncategorized`.
- Remove currency symbols and cast price, cost, and inventory.
- Estimate missing cost at 65% of price.
- Reject nonpositive prices, negative costs/inventory, and orphan store keys.

In [ ]:
products_stage = (
    bronze["products"].dropDuplicates(["ProductID"])
    .withColumn("Category", F.initcap(F.trim(F.coalesce(F.col("Category"), F.lit("Uncategorized")))))
    .withColumn("UnitPrice", numeric_from_raw("UnitPriceRaw"))
    .withColumn("UnitCost", numeric_from_raw("UnitCostRaw"))
    .withColumn("InventoryQty", numeric_from_raw("InventoryQtyRaw").cast("int"))
    .withColumn("UnitCost", F.coalesce(F.col("UnitCost"), F.round(F.col("UnitPrice") * 0.65, 2)))
    .filter((F.col("UnitPrice") > 0) & (F.col("UnitCost") >= 0) & (F.col("InventoryQty") >= 0))
)
products_silver = (
    products_stage.join(stores_silver.select("StoreID"), "StoreID", "inner")
    .withColumn("EstimatedMarginPct", F.round((F.col("UnitPrice") - F.col("UnitCost")) / F.col("UnitPrice") * 100, 2))
    .select("ProductID", "StoreID", "ProductName", "Category", "UnitPrice", "UnitCost", "InventoryQty", "EstimatedMarginPct")
)
write_silver(products_silver, "products")
display(products_silver.groupBy("Category").count().orderBy(F.desc("count")))

## 5. Orders - owner: Sweta

- Normalize status and payment method.
- Parse both timestamp formats.
- Remove orders with invalid customer/store keys.
- Add calendar attributes for dashboard analysis.

In [ ]:
orders_stage = (
    bronze["orders"].dropDuplicates(["OrderID"])
    .withColumn("OrderStatus", F.initcap(F.trim(F.col("OrderStatus"))))
    .withColumn("PaymentMethod", F.initcap(F.trim(F.col("PaymentMethod"))))
    .withColumn("OrderTimestamp", F.coalesce(
        F.to_timestamp("OrderTimestampRaw", "yyyy-MM-dd HH:mm:ss"),
        F.to_timestamp("OrderTimestampRaw", "MM/dd/yyyy HH:mm"),
    ))
    .filter(F.col("OrderTimestamp").isNotNull())
)
orders_silver = (
    orders_stage
    .join(customers_silver.select("CustomerID"), "CustomerID", "inner")
    .join(stores_silver.select("StoreID"), "StoreID", "inner")
    .withColumn("OrderDate", F.to_date("OrderTimestamp"))
    .withColumn("OrderYear", F.year("OrderTimestamp"))
    .withColumn("OrderMonth", F.month("OrderTimestamp"))
    .withColumn("YearMonth", F.date_format("OrderTimestamp", "yyyy-MM"))
    .withColumn("DayName", F.date_format("OrderTimestamp", "EEEE"))
    .withColumn("IsWeekend", F.dayofweek("OrderTimestamp").isin([1, 7]))
    .select("OrderID", "CustomerID", "StoreID", "OrderTimestamp", "OrderDate", "OrderYear", "OrderMonth", "YearMonth", "DayName", "IsWeekend", "OrderStatus", "PaymentMethod")
)
write_silver(orders_silver, "orders")
display(orders_silver.groupBy("OrderStatus").count().orderBy(F.desc("count")))

## 6. Order items - owner: Omar Leopoldo

- Cast quantity and price after removing nonnumeric characters.
- Standardize discount formats to decimals.
- Reject invalid quantities/prices/discounts and orphan keys.
- Enforce that every product belongs to the same store as its order.

In [ ]:
discount_number = numeric_from_raw("DiscountRaw")
items_stage = (
    bronze["order_items"].dropDuplicates(["OrderItemID"])
    .withColumn("Quantity", numeric_from_raw("QuantityRaw").cast("int"))
    .withColumn("UnitPrice", numeric_from_raw("UnitPriceRaw"))
    .withColumn("DiscountPct", F.when(discount_number > 1, discount_number / 100.0).otherwise(discount_number))
    .filter(
        (F.col("Quantity") > 0) & (F.col("UnitPrice") > 0)
        & F.col("DiscountPct").between(0, 0.60)
    )
)
item_integrity = (
    items_stage
    .join(orders_silver.select("OrderID", F.col("StoreID").alias("OrderStoreID")), "OrderID", "inner")
    .join(products_silver.select("ProductID", F.col("StoreID").alias("ProductStoreID")), "ProductID", "inner")
    .filter(F.col("OrderStoreID") == F.col("ProductStoreID"))
)
order_items_silver = item_integrity.select(
    "OrderItemID", "OrderID", "ProductID", "Quantity", "UnitPrice", "DiscountPct"
)
write_silver(order_items_silver, "order_items")
display(order_items_silver.describe(["Quantity", "UnitPrice", "DiscountPct"]))

## 7. Deliveries - owner: Shreyansh Pankaj

- Normalize delivery status.
- Remove units and cast distance/minutes to numeric.
- Reject negative distance or promised time and orphan orders.
- Derive delay minutes and an on-time indicator only for completed deliveries.

In [ ]:
deliveries_stage = (
    bronze["deliveries"].dropDuplicates(["DeliveryID"])
    .withColumn("DeliveryStatus", F.initcap(F.trim(F.col("DeliveryStatus"))))
    .withColumn("DistanceKm", numeric_from_raw("DistanceKmRaw"))
    .withColumn("PromisedMinutes", numeric_from_raw("PromisedMinutesRaw"))
    .withColumn("ActualMinutes", numeric_from_raw("ActualMinutesRaw"))
    .filter((F.col("DistanceKm") > 0) & (F.col("PromisedMinutes") > 0))
)
deliveries_silver = (
    deliveries_stage.join(orders_silver.select("OrderID"), "OrderID", "inner")
    .withColumn("DelayMinutes", F.when(
        F.col("ActualMinutes").isNotNull(), F.greatest(F.col("ActualMinutes") - F.col("PromisedMinutes"), F.lit(0.0))
    ))
    .withColumn("IsOnTime", F.when(
        (F.col("DeliveryStatus") == "Delivered") & F.col("ActualMinutes").isNotNull(),
        F.col("ActualMinutes") <= F.col("PromisedMinutes")
    ))
    .select("DeliveryID", "OrderID", "DriverID", "DriverName", "DeliveryStatus", "DistanceKm", "PromisedMinutes", "ActualMinutes", "DelayMinutes", "IsOnTime")
)
write_silver(deliveries_silver, "deliveries")
display(deliveries_silver.groupBy("DeliveryStatus").count().orderBy(F.desc("count")))

## 8. Data-quality summary and integrity assertions

In [ ]:
silver = {
    "customers": customers_silver, "stores": stores_silver, "products": products_silver,
    "orders": orders_silver, "order_items": order_items_silver, "deliveries": deliveries_silver,
}
quality_rows = []
for name in bronze_names:
    before = bronze[name].count()
    after = silver[name].count()
    quality_rows.append((name, before, after, before - after, round((before - after) * 100.0 / before, 2)))

quality_summary = spark.createDataFrame(
    quality_rows, ["table_name", "bronze_rows", "silver_rows", "rows_removed", "removed_pct"]
)
(quality_summary.write.format("delta").mode("overwrite")
 .option("overwriteSchema", "true").saveAsTable(f"{CATALOG}.{SCHEMA}.dq_summary_silver"))
display(quality_summary.orderBy("table_name"))

assert duplicate_count(customers_silver, "CustomerID") == 0
assert duplicate_count(stores_silver, "StoreID") == 0
assert duplicate_count(products_silver, "ProductID") == 0
assert duplicate_count(orders_silver, "OrderID") == 0
assert duplicate_count(order_items_silver, "OrderItemID") == 0
assert duplicate_count(deliveries_silver, "DeliveryID") == 0
assert orders_silver.join(customers_silver, "CustomerID", "left_anti").count() == 0
assert orders_silver.join(stores_silver, "StoreID", "left_anti").count() == 0
assert order_items_silver.join(orders_silver, "OrderID", "left_anti").count() == 0
assert order_items_silver.join(products_silver, "ProductID", "left_anti").count() == 0
assert deliveries_silver.join(orders_silver, "OrderID", "left_anti").count() == 0
print("All Silver primary-key and foreign-key checks passed. Continue with 03_gold_eda.ipynb.")